<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مقدمه و هدف پروژه
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این پروژه، هدف ما پیاده‌سازی یک مدل طبقه‌بندی‌کننده برای شناسایی پیام‌های اسپم و غیر اسپم (ham) است.
برای این کار از داده‌های متنی استفاده می‌کنیم که شامل پیام‌های متنی با برچسب‌های "ham" و "spam" می‌باشند.
ما قرار است از مدل Naive Bayes برای این کار استفاده کنیم که در این فصل یاد گرفتید.
</font>
</p>


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش کتابخانه‌های مورد نیاز برای انجام پروژه را وارد می‌کنیم.
</font>
</p>


In [ ]:
import pandas as pd
import numpy as np
import re
import string
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
بارگذاری داده‌ها
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش داده‌ها از فایل CSV بارگذاری می‌شوند و تنها ستون‌های مورد نیاز انتخاب و نام‌گذاری می‌شوند.
</font>
</p>


In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
نمایش اولین 5 ردیف داده
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
با استفاده از این کد، اولین 5 ردیف از داده‌ها برای بررسی بارگذاری صحیح آن‌ها نمایش داده می‌شود.
</font>
</p>


In [ ]:
df.head()

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
کدگذاری برچسب‌ها
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش، برچسب‌های 'ham' و 'spam' باید به مقادیر عددی 0 و 1 تغییر داده شوند تا برای مدل‌سازی قابل استفاده باشند.
</font>
</p>


In [ ]:
# TODO
df['label'] = None
df.head()

In [ ]:
# کدگذاری برچسب‌ها - تبدیل ham به 0 و spam به 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
توابع پیش‌پردازش متن
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این قسمت، متنی که از داده‌ها استخراج می‌شود، به حروف کوچک تبدیل شده و اعداد و علائم نگارشی از آن حذف می‌شوند. سپس متن به کلمات جداگانه تقسیم می‌شود.
</font>
</p>


In [ ]:
# TODO

def preprocess_text(text):
    text = None
    text = None
    text = None
    return text.split()

df['processed_text'] = df['text'].apply(preprocess_text)
df.head()

In [ ]:
def preprocess_text(text):
    text = text.lower()  # تبدیل به حروف کوچک
    text = re.sub(r'\d+', '', text)  # حذف اعداد
    text = text.translate(str.maketrans('', '', string.punctuation))  # حذف علائم نگارشی
    return text.split()

df['processed_text'] = df['text'].apply(preprocess_text)
df.head()

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تقسیم داده‌ها به مجموعه‌های آموزشی و آزمایشی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش، داده‌ها به دو مجموعه آموزشی (80٪ داده‌ها) و آزمایشی (20٪ داده‌ها) تقسیم می‌شوند.
</font>
</p>


In [ ]:
train_size = int(0.8 * len(df))
train_data = df[:train_size]
test_data = df[train_size:]

In [ ]:
train_size = int(0.8 * len(df))
train_data = df[:train_size]
test_data = df[train_size:]

print(f"تعداد کل داده‌ها: {len(df)}")
print(f"تعداد داده‌های آموزشی: {len(train_data)}")
print(f"تعداد داده‌های آزمایشی: {len(test_data)}")

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پیاده‌سازی مدل بیز ساده
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این تمرین، شما باید مدل Naive Bayes را برای پیش‌بینی برچسب‌ها پیاده‌سازی کنید. هدف این است که بر اساس داده‌های آموزشی، مدل را برای یادگیری احتمالات کلاس‌ها و واژه‌ها آموزش داده و سپس بر اساس داده‌های جدید، کلاس‌های مربوطه را پیش‌بینی کنید.
</font>
</p>

---

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
مراحل تمرین
</h3>

<ul dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<li>در تابع <code>train</code>، باید اطلاعات مربوط به احتمال کلاس‌ها و احتمال وقوع واژه‌ها در هر کلاس محاسبه شود. از شمارش واژه‌ها و فرمول احتمال شرطی استفاده کنید.</li>
<li>در تابع <code>predict</code>، با استفاده از داده‌های جدید و احتمالاتی که در مرحله آموزش محاسبه شده، برچسب هر نمونه پیش‌بینی شود.</li>
<li>برای جلوگیری از مشکلات ناشی از صفر شدن احتمال‌ها، از تکنیک هموارسازی لاپلاس استفاده کنید.</li>
</ul>


In [ ]:
class NaiveBayes:
    def __init__(self):
        """
        Initialize the model with variables to store class probabilities,
        word counts per class, and the vocabulary.
        """
        self.class_word_counts = defaultdict(Counter)  # Stores word counts for each class
        self.class_totals = defaultdict(int)           # Total word count for each class
        self.class_probs = defaultdict(float)          # Prior probabilities of each class
        self.vocab = set()                             # Vocabulary of the training data

    def train(self, X, y):
        """
        Train the Naive Bayes model using the training data.

        Parameters:
        X: List of sentences/documents (tokenized)
        y: List of corresponding class labels

        Steps:
        1. Count the number of samples for each class.
        2. Calculate the prior probabilities for each class.
        3. Count the occurrences of each word in each class.
        4. Update the vocabulary with all words seen in the training data.
        """
        # TODO: Count the number of samples in each class
        # class_counts = Counter(y)

        # TODO: Calculate prior probabilities for each class
        # for cls, count in class_counts.items():
        #     self.class_probs[cls] = ...

        # TODO: Count words for each class and update vocabulary
        # for text, cls in zip(X, y):
        #     for word in text:
        #         ...
        pass

    def predict(self, X):
        """
        Predict the class labels for a list of input samples.

        Parameters:
        X: List of tokenized documents

        Returns:
        List of predicted class labels for each sample in X.
        """
        predictions = []
        vocab_size = len(self.vocab)  # Size of the vocabulary

        for text in X:
            class_scores = {}

            # TODO: Compute the score for each class
            # for cls in self.class_probs:
            #     class_scores[cls] = ...
            #     for word in text:
            #         ...

            # TODO: Select the class with the highest score
            # predictions.append(...)
        return predictions


In [ ]:
class NaiveBayes:
    def __init__(self):
        """
        Initialize the model with variables to store class probabilities,
        word counts per class, and the vocabulary.
        """
        self.class_word_counts = defaultdict(Counter)  # Stores word counts for each class
        self.class_totals = defaultdict(int)           # Total word count for each class
        self.class_probs = defaultdict(float)          # Prior probabilities of each class
        self.vocab = set()                             # Vocabulary of the training data

    def train(self, X, y):
        """
        Train the Naive Bayes model using the training data.
        """
        # Count the number of samples in each class
        class_counts = Counter(y)
        total_samples = len(y)

        # Calculate prior probabilities for each class
        for cls, count in class_counts.items():
            self.class_probs[cls] = count / total_samples

        # Count words for each class and update vocabulary
        for text, cls in zip(X, y):
            for word in text:
                self.class_word_counts[cls][word] += 1
                self.class_totals[cls] += 1
                self.vocab.add(word)

    def predict(self, X):
        """
        Predict the class labels for a list of input samples.
        """
        predictions = []
        vocab_size = len(self.vocab)  # Size of the vocabulary

        for text in X:
            class_scores = {}

            # Compute the score for each class
            for cls in self.class_probs:
                class_scores[cls] = np.log(self.class_probs[cls])
                for word in text:
                    # Laplace smoothing: (word_count + 1) / (total_words + vocab_size)
                    word_prob = (self.class_word_counts[cls][word] + 1) / (self.class_totals[cls] + vocab_size)
                    class_scores[cls] += np.log(word_prob)

            # Select the class with the highest score
            predictions.append(max(class_scores, key=class_scores.get))

        return predictions

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
آموزش مدل Naive Bayes
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش، مدل Naive Bayes با استفاده از داده‌های آموزشی (`X_train`, `y_train`) آموزش داده می‌شود.
</font>
</p>


In [ ]:
X_train = train_data['processed_text']
y_train = train_data['label']
X_test = test_data['processed_text']
y_test = test_data['label']

nb_model = NaiveBayes()
nb_model.train(X_train, y_train)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پیش‌بینی برچسب‌های مجموعه آزمایشی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش، از مدل آموزش‌دیده‌ی Naive Bayes برای پیش‌بینی برچسب‌های مجموعه آزمایشی استفاده می‌شود. متغیر X_test شامل داده‌های متنی تست است که به مدل داده می‌شود تا برچسب‌های مربوط به آن‌ها (آیا پیام اسپم است یا خیر) پیش‌بینی شوند. خروجی این تابع، لیستی از پیش‌بینی‌ها برای هر نمونه در مجموعه آزمایشی است که در متغیر y_pred ذخیره می‌شود.
</font>
</p>


In [ ]:
y_pred = nb_model.predict(X_test)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
ارزیابی مدل با استفاده از معیارهای دقت، فراخوانی، F1 و دقت کلی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش، عملکرد مدل پیش‌بینی با استفاده از معیارهای مختلف ارزیابی می‌شود. معیارهای ارزیابی شامل دقت (<i>Precision</i>)، فراخوانی (<i>Recall</i>)، امتیاز F1 و دقت کلی (<i>Accuracy</i>) هستند.
</font>
</p>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
برای این منظور، باید تعداد مثبت‌های واقعی (<i>True Positives</i>)، منفی‌های واقعی (<i>True Negatives</i>)، مثبت‌های کاذب (<i>False Positives</i>) و منفی‌های کاذب (<i>False Negatives</i>) را محاسبه کنید. سپس از فرمول‌های مناسب برای محاسبه معیارهای ارزیابی استفاده کنید.
</font>
</p>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
راهنما:
</font>
</p>

<ul dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
    <li>برای محاسبه <code>TP</code>، <code>TN</code>، <code>FP</code> و <code>FN</code> از مقایسه مقادیر واقعی (<code>y_true</code>) و پیش‌بینی‌شده (<code>y_pred</code>) استفاده کنید.</li>
    <li>اگر تعداد نمونه‌ها در مخرج صفر باشد، مقدار معیار را برابر صفر در نظر بگیرید تا از خطای تقسیم بر صفر جلوگیری شود.</li>
</ul>


In [ ]:
# TODO

def evaluate(y_true, y_pred):
    tp = None  # True Positives
    tn = None  # True Negatives
    fp = None  # False Positives
    fn = None  # False Negatives

    precision = None
    recall = None
    f1 = None
    accuracy = None

    return precision, recall, f1, accuracy

y_test_np = y_test.values
y_pred_np = np.array(y_pred)

precision, recall, f1, accuracy = evaluate(y_test_np, y_pred_np)

print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))
print("F1 Score:", round(f1, 2))
print("Accuracy:", round(accuracy, 2))

In [ ]:
def evaluate(y_true, y_pred):
    tp = sum((y_true == 1) & (y_pred == 1))  # True Positives
    tn = sum((y_true == 0) & (y_pred == 0))  # True Negatives
    fp = sum((y_true == 0) & (y_pred == 1))  # False Positives
    fn = sum((y_true == 1) & (y_pred == 0))  # False Negatives

    # محاسبه Precision
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    # محاسبه Recall
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    # محاسبه F1 Score
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # محاسبه Accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0

    return precision, recall, f1, accuracy

y_test_np = y_test.values
y_pred_np = np.array(y_pred)

precision, recall, f1, accuracy = evaluate(y_test_np, y_pred_np)

print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))
print("F1 Score:", round(f1, 2))
print("Accuracy:", round(accuracy, 2))

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
نمایش ماتریس اشتباهات (Confusion Matrix)
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این بخش، ماتریس (Confusion Matrix) .برای ارزیابی عملکرد مدل رسم می‌شود. این ماتریس چهار مقدار اصلی را نشان می‌دهد که بیانگر نحوه عملکرد مدل در این تسک میباشد.
</font>
</p>


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_np, y_pred_np)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_np, y_pred_np)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
ذخیره نتایج پیش‌بینی و فشرده‌سازی فایل‌ها برای ارسال
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این قسمت، با اجرای سل زیر، یک فایل زیپ ایجاد میشود که آن را جهت نمره دهی در سامانه آپلود نمایید.
</font>
</p>


In [ ]:
import zipfile

pred_df = pd.DataFrame({'id': test_data.index, 'predicted_label': y_pred})
pred_df['predicted_label'] = pred_df['predicted_label'].map({0: 'ham', 1: 'spam'})
pred_df.to_csv('predictions.csv', index=False)

files_to_zip = ['spam_detection.ipynb', 'predictions.csv']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        try:
            zipf.write(file)
            print(f"Added {file} to {zip_filename}")
        except FileNotFoundError:
            print(f"Warning: {file} not found. Skipping.")

print(f"Files have been zipped into {zip_filename}, you can now submit it!")

